# Day 2: Library Helpdesk Chatbot
Topics covered: system prompt (Role, Fact, Rule, Format), few-shot prompting, JSON output, memory by hand, Python basics.

**Before running:** add your key in Colab **Secrets** (key icon, left sidebar) with the name `GROQ_API_KEY` and turn on **Notebook access**.

## 1. Install and set up the LLM

In [ ]:
!pip install -q langchain_groq groq

In [ ]:
import os
import json
from google.colab import userdata
from langchain_groq import ChatGroq

# try/except: show a helpful message instead of crashing if the key is missing
try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    print("Please add GROQ_API_KEY to your Colab Secrets (key icon in the left sidebar).")

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

## 2. System prompt: Role, Fact, Rule, Format
Topic: a **college library helpdesk**. Change the facts to match your own topic if you like.

In [ ]:
SYSTEM = """ROLE: You are the helpdesk assistant for the Central Library of a college. You help students.
FACTS: The library is open 9 am to 6 pm, Monday to Saturday, and closed on Sunday.
A student can borrow up to 3 books for 14 days. A book can be renewed once, online.
The late fine is Rs 2 per day per book. A college ID card is needed for membership.
Photocopy costs Rs 1 per page and printout costs Rs 2 per page.
RULES: Never promise that a book is available; say "I'll check with the librarian".
If you do not know something, say so. Do not make up facts or share personal details of students.
FORMAT: Reply politely and in a friendly tone, in under 50 words, in the same language the student uses."""

## 3. Ask function (single question, no memory)

In [ ]:
def ask(question: str) -> str:
    """Send one question to the LLM with the system prompt and return the reply text."""
    return llm.invoke([("system", SYSTEM), ("human", question)]).content

print(ask("What time does the library open on Saturday?"))
print(ask("Can you reserve Python Crash Course for me tomorrow?"))

## 4. Few-shot prompting
We give the model a few examples so it labels messages consistently: `reservation`, `question` or `complaint`.
This does not train the model. It only shows reference examples before answering.

In [ ]:
FEW_SHOT_SYSTEM = "Classify the student's message as exactly one word: reservation, question or complaint."

examples = [
    ("human", "Please keep Data Structures by Karumanchi for me till Friday"),
    ("ai", "reservation"),
    ("human", "Can I renew a book twice?"),
    ("ai", "question"),
    ("human", "The fans in the reading hall are not working"),
    ("ai", "complaint"),
]

def classify(text: str) -> str:
    """Return the label for a student message using few-shot examples."""
    messages = [("system", FEW_SHOT_SYSTEM)] + examples + [("human", text)]
    return llm.invoke(messages).content.strip().lower()

for msg in ["I need two copies of Operating Systems for Monday",
            "Do you have a printing facility?",
            "The librarian was rude to me yesterday"]:
    print(msg, "->", classify(msg))

## 5. JSON output
JSON is easy to store and calculate with. `json.loads` turns a JSON string into a Python dictionary and `json.dumps` turns a dictionary into a JSON string.
If the model replies "Sure, here is JSON", `json.loads` fails with a `JSONDecodeError`, so we tell it to return only JSON.

In [ ]:
JSON_SYSTEM = """Extract a book request from the message and return ONLY valid JSON, with no extra text.
Keys: "student_name" (string), "book_title" (string), "copies" (number), "urgent" (true or false).
Use double quotes and lowercase true/false."""

def extract_request(text: str) -> dict:
    """Return the book request as a Python dictionary, or an empty dict if the reply is not valid JSON."""
    raw = llm.invoke([("system", JSON_SYSTEM), ("human", text)]).content
    raw = raw.replace("```json", "").replace("```", "").strip()  # remove code fences if present
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        print("Reply was not valid JSON:", raw)
        return {}

request = extract_request("Hi, I am Ravi. I need 2 copies of Engineering Mathematics urgently.")
print(request)
print(json.dumps(request, indent=2))

## 6. Python basics for agents: dictionaries, type hints, try/except

In [ ]:
services = {"photocopy": 1, "printout": 2}   # price per page in rupees

def service_cost(service: str, pages: int) -> int:
    """Return the total cost. services.get(name, 0) gives 0 if the service is missing."""
    return services.get(service, 0) * pages

def late_fine(days_late: int, books: int = 1) -> int:
    """Late fine is Rs 2 per day per book."""
    return 2 * days_late * books

print(service_cost("photocopy", 20))   # 20
print(service_cost("scan", 5))         # 0 because scan is not in the dictionary
print(late_fine(5, 2))                 # 20

## 7. Memory by hand + chatbot loop
Each new question is sent along with the previous messages so the bot remembers the context.
Long histories cost more tokens, so we keep only the last `MAX_TURNS` turns (one turn = one question + one answer).
Type `bye`, `exit` or `quit` to stop.

In [ ]:
MAX_TURNS = 6
history = []   # list of (role, text) tuples

def chat(question: str) -> str:
    """Answer a question using the system prompt plus the last MAX_TURNS turns of history."""
    history.append(("human", question))
    recent = history[-MAX_TURNS * 2:]
    reply = llm.invoke([("system", SYSTEM)] + recent).content
    history.append(("ai", reply))
    return reply

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ("bye", "exit", "quit"):
        print("Bot: Goodbye! Happy reading.")
        break
    try:
        print("Bot:", chat(user_input))
    except Exception as e:
        print("Bot: Sorry, something went wrong connecting to the model.", e)

## 8. Experiments to try (for learning)
1. Delete the **FORMAT** line from `SYSTEM` and see how long the answers become.
2. Delete the **RULES** line and ask "Can you reserve a book for tomorrow?"
3. Delete the **FACTS** line and ask about the late fine.
4. Change `MAX_TURNS` to 1 and tell the bot your name, then ask it a few questions and see if it forgets.